# Heart Disease Prediction - Random Forest Classifier (Day 5)

This notebook covers the training, evaluation, and feature importance analysis of a Random Forest Classifier trained on the preprocessed combined UCI Heart Disease dataset.

## Step 1: Import Libraries

**What & Why:**
- **What:** Imported data handling, plotting, and Random Forest ensemble model classes.
- **Why:** These provide the necessary modules for building the ensemble classifier, partition validation, and plotting metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

## Step 2: Load Dataset

**What & Why:**
- **What:** Loaded `heart_processed.csv` from the `data/` directory.
- **Why:** To read the fully cleaned and encoded dataset to train the ensemble model.

In [ ]:
df = pd.read_csv("../data/heart_processed.csv")
df.head()

## Step 3: Split Features and Target

**What & Why:**
- **What:** Separated the clean columns into predictors matrix `X` and output label vector `y`.
- **Why:** Required input formatting to pass arguments to splitting and fitting algorithms.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

## Step 4: Train-Test Split

**What, Why & Observations:**
- **What:** Split features and target into 80% training and 20% testing sets with stratification.
- **Why:** Like standard decision trees, **Random Forest does not require feature scaling** because node splits are determined relative to single features individually, which is scale-invariant.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 5: Create the Model

**What & Why:**
- **What:** Instantiated the `RandomForestClassifier` ensemble model.
- **Why:** Setting `n_estimators=100` trains 100 independent decision trees to vote on predictions, and `random_state=42` ensures consistent bootstrapping results.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

## Step 6: Train the Model

**What & Why:**
- **What:** Trained the Random Forest Classifier on training feature data.
- **Why:** To fit 100 bootstrapped decision trees on the training records, preparing the voting ensemble.

In [ ]:
rf.fit(X_train, y_train)

## Step 7: Make Predictions

**What & Why:**
- **What:** Predicted disease outcomes on test feature split (`X_test`).
- **Why:** To output predictions to evaluate against the actual labels (`y_test`).

In [ ]:
y_pred = rf.predict(X_test)

## Step 8: Calculate Accuracy

**What, Why & Observations:**
- **What:** Computed test accuracy score.
- **Why:** Baseline correctness measurement.
- **Observations:** The Random Forest Classifier achieved **82.61% accuracy**, which is a significant improvement over the **76.09%** of a single Decision Tree, and slightly higher than the **82.07%** of Logistic Regression.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

## Step 9: Confusion Matrix

**What, Why & Observations:**
- **What:** Plotted the confusion matrix and saved it to the `images/` directory.
- **Why:** To review classification errors. In medical diagnostics, minimizing false negatives is critical.
- **Observations:** 
  - 63 patients were correctly predicted as healthy (True Negatives).
  - 89 patients were correctly predicted as diseased (True Positives).
  - 19 patients were incorrectly predicted as diseased (False Positives).
  - 13 patients were incorrectly predicted as healthy (False Negatives).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Purples")
plt.title("Random Forest Confusion Matrix")
plt.savefig("../images/random_forest_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 10: Classification Report

**What, Why & Observations:**
- **What:** Generated precision, recall, and F1-score details.
- **Why:** Clinical validation of model safety (specifically looking for high recall).
- **Observations:** The ensemble model performs balanced across both classes:
  - **Class 0 (Healthy):** F1-score: **0.80** (Precision: 0.83, Recall: 0.77)
  - **Class 1 (Disease):** F1-score: **0.85** (Precision: 0.82, Recall: 0.87)

In [ ]:
print(classification_report(y_test, y_pred))

## Step 11: Feature Importance

**What, Why & Observations:**
- **What:** Computed Mean Decrease in Impurity (MDI) feature importances.
- **Why:** To rank features based on split contribution across all 100 trees.
- **Observations:** Chest pain (`cp`) has the highest score (0.151), followed closely by cholesterol (`chol`, 0.148) and maximum heart rate (`thalch`, 0.134).

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})
importance = importance.sort_values(
    by="Importance",
    ascending=True
)
importance

## Step 12: Visualize Feature Importance

**What & Why:**
- **What:** Rendered and saved a horizontal bar plot of the features sorted by Gini importance.
- **Why:** To visualize the relative importance of clinical predictors clearly.

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(
    importance["Feature"],
    importance["Importance"]
)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.savefig("../images/random_forest_importance.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 13: Experiment with Hyperparameters

**What, Why & Observations:**
- **What:** Trained and evaluated Random Forest models with 50 and 200 trees.
- **Why:** To observe how changing the ensemble size impacts accuracy performance.
- **Observations:** 
  - Accuracy with **50 Trees**: **83.70%**
  - Accuracy with **100 Trees**: **82.61%**
  - Accuracy with **200 Trees**: **83.15%**
  - Increasing the number of trees does not improve accuracy linearly. 50 trees achieved the best score on this dataset, demonstrating that larger ensembles do not always guarantee performance gains.

In [ ]:
# 50 Trees
rf_50 = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)
rf_50.fit(X_train, y_train)
print("Accuracy (50 Trees):", accuracy_score(y_test, rf_50.predict(X_test)))

# 200 Trees
rf_200 = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf_200.fit(X_train, y_train)
print("Accuracy (200 Trees):", accuracy_score(y_test, rf_200.predict(X_test)))

## Step 14: Compare All Models

**What & Why:**
- **What:** Contrast results across all classification models tested.
- **Why:** To make an informed choice on the best candidate model for deployment.

### Performance Comparison Table

| Model | Accuracy | Precision (Class 0 / 1) | Recall (Class 0 / 1) | F1-Score (Class 0 / 1) |
|---|---|---|---|---|
| **Logistic Regression** | **82.07%** | 0.80 / 0.83 | 0.79 / 0.84 | 0.80 / 0.84 |
| **Decision Tree (Unpruned)** | **76.09%** | 0.74 / 0.77 | 0.71 / 0.80 | 0.72 / 0.79 |
| **Decision Tree (max_depth=4)** | **80.43%** | 0.80 / 0.81 | 0.74 / 0.85 | 0.77 / 0.83 |
| **Random Forest (100 Trees)** | **82.61%** | 0.83 / 0.82 | 0.77 / 0.87 | 0.80 / 0.85 |
| **Random Forest (50 Trees)** | **83.70%** | 0.80 / 0.87 | 0.84 / 0.83 | 0.82 / 0.85 |

## 📝 Mini Exercise Questions & Answers

**1. What accuracy did Random Forest achieve?**
- The standard 100-tree Random Forest Classifier achieved **82.61%** accuracy, while the 50-tree model achieved **83.70%** accuracy.

**2. Did it outperform Logistic Regression?**
- Yes, the Random Forest (both the 50-tree model at **83.70%** and the 100-tree model at **82.61%**) outperformed Logistic Regression (**82.07%**).

**3. Did it outperform Decision Tree?**
- Yes, significantly. The unpruned Decision Tree achieved only **76.09%** accuracy and the pruned depth-limited tree achieved **80.43%**, whereas Random Forest achieved up to **83.70%**.

**4. Which feature was the most important?**
- Chest pain type (`cp`) was the most important feature (MDI score **0.151**), followed closely by cholesterol (`chol` at **0.148**).

**5. Why does Random Forest reduce overfitting?**
- Random Forest trains multiple independent decision trees using **bagging** (bootstrap aggregation) and **feature randomness** (only a subset of features is evaluated at each node split). Averaging predictions across these diverse, uncorrelated trees cancels out individual tree variance, preventing overfitting.

**6. Does increasing the number of trees always improve accuracy?**
- No. Beyond a certain threshold (e.g. 50-100 trees for this size of dataset), the model converges. Adding more trees does not improve performance but increases computational overhead.